# Protocol

> When messages are transmitted: queues, time slots and medium access.

In [ ]:
#| default_exp protocol

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

from __future__ import annotations

from abc import ABC, abstractmethod
from collections import defaultdict, deque
from typing import Collection, Deque, Dict, List, Sequence

import numpy as np

from comm_core.core import Message, NodeId, Transmission, TransmissionResult



In [ ]:
#| export
class Protocol(ABC):

    @abstractmethod
    def submit(
        self,
        message: Message,
        time: float,
    ) -> None:
        pass

    def reset(self, seed: int | None = None) -> None:
        "Drop the queued messages (e.g. a new episode); `seed` reseeds the protocol's random generator, if it has one."
        pass

    def feedback(
        self,
        results: List[TransmissionResult], # The results of the transmissions just scheduled
    ) -> None:
        "What happened to the transmissions of the last step (an acknowledgement), e.g. to retransmit the lost ones."
        pass

    @abstractmethod
    def schedule(
        self,
        time: float,
    ) -> List[Transmission]:
        pass

In [ ]:
#| export
class IdealProtocol(Protocol):

    def __init__(self):
        self._queue: Deque[Message] = deque()

    def reset(self, seed: int | None = None) -> None:
        self._queue.clear()

    def submit(
        self,
        message: Message,
        time: float,
    ) -> None:
        self._queue.append(message)

    def schedule(
        self,
        time: float,
    ) -> List[Transmission]:

        transmissions = []

        while self._queue:

            message = self._queue.popleft()

            transmissions.append(
                Transmission(
                    message=message,
                    sender=message.sender,
                    receiver=message.receiver,
                    start_time=time,
                    end_time=time,
                )
            )

        return transmissions
    

## Slotted protocols

`IdealProtocol` sends everything at once. With `InterferenceBackend` (see `system`), a `Network` step's
transmissions happen at the same time and interfere, and half-duplex nodes that transmit cannot receive. If
every node talks at every step, nobody hears. A medium access (MAC) protocol decides who transmits when.

In a `SlottedProtocol`, every `Network` step is a time slot.

* **Who sends.** In each slot, `senders` picks the nodes that transmit, and each of them sends all its waiting
  messages (one radio, so a sender's messages don't interfere with each other).
* **Waiting.** The other nodes' messages wait for a later slot. `queue_size` keeps only a sender's newest
  messages: in multi-agent learning, old messages are often stale.
* **Retransmissions.** With `max_retries`, the `Network` reports the results back (`feedback`, an
  acknowledgement), and lost messages are sent again in the sender's next slot.

Two protocols are built on it:

* `TDMAProtocol` gives each node its own slots, in a repeating frame, so there are no collisions. Nodes far enough
  apart can share a slot (spatial reuse).
* `SlottedALOHA` lets each node with waiting messages transmit with probability `p` in every slot, without
  coordination. Collisions happen, and the `InterferenceBackend` decides which transmissions survive them.

In [ ]:
#| export
class SlottedProtocol(Protocol):
    "Each network step is a slot; `senders` decides which nodes transmit in it, and the others' messages wait."

    def __init__(
        self,
        queue_size: int | None = None, # Messages waiting per sender; beyond it the oldest are dropped (None: no limit)
        max_retries: int = 0, # Retransmissions of a lost message (from the network's feedback)
    ):
        self.queue_size, self.max_retries = queue_size, max_retries
        self._queues: Dict[NodeId, Deque[Message]] = defaultdict(deque)
        self._attempts: Dict[str, int] = {}
        self.slot = 0
        self.dropped = 0                               # messages dropped because a queue was full

    @abstractmethod
    def senders(
        self,
        slot: int, # Index of the slot (0, 1, ...)
        waiting: List[NodeId], # The nodes with messages waiting
    ) -> Collection[NodeId]:
        "The nodes that transmit in this slot."

    def reset(self, seed: int | None = None) -> None:
        self._queues.clear()
        self._attempts.clear()
        self.slot = 0
        self.dropped = 0

    def submit(self, message: Message, time: float) -> None:
        queue = self._queues[message.sender]
        queue.append(message)
        if self.queue_size is not None and len(queue) > self.queue_size:
            dropped = queue.popleft()
            self._attempts.pop(dropped.id, None)
            self.dropped += 1

    def waiting(self, node: NodeId) -> int:
        "Number of messages `node` has waiting."
        return len(self._queues.get(node, ()))

    def schedule(self, time: float) -> List[Transmission]:
        waiting = [node for node, queue in self._queues.items() if queue]
        chosen = set(self.senders(self.slot, waiting))
        self.slot += 1
        transmissions = []
        for node in waiting:
            if node not in chosen:
                continue
            queue = self._queues[node]
            while queue:
                message = queue.popleft()
                transmissions.append(Transmission(message=message, sender=message.sender, receiver=message.receiver,
                                                  start_time=time, end_time=time))
        return transmissions

    def feedback(self, results: List[TransmissionResult]) -> None:
        "Lost messages go back to the front of their sender's queue, at most `max_retries` times."
        if not self.max_retries:
            return
        retry = []
        for result in results:
            message = result.transmission.message
            if result.success:
                self._attempts.pop(message.id, None)
                continue
            attempts = self._attempts.get(message.id, 0) + 1
            if attempts <= self.max_retries:
                self._attempts[message.id] = attempts
                retry.append(message)
            else:
                self._attempts.pop(message.id, None)
        for message in reversed(retry):
            self._queues[message.sender].appendleft(message)


class TDMAProtocol(SlottedProtocol):
    "Time division: each slot of a repeating frame belongs to some nodes, which alone transmit in it."

    def __init__(
        self,
        frame: Sequence, # Owners of each slot: node ids (one per slot), or collections of node ids sharing a slot
        queue_size: int | None = None, # Messages waiting per sender (see `SlottedProtocol`)
        max_retries: int = 0, # Retransmissions of a lost message
    ):
        super().__init__(queue_size, max_retries)
        self.frame = [{owner} if isinstance(owner, str) else set(owner) for owner in frame]
        if not self.frame:
            raise ValueError("the frame needs at least one slot")

    def senders(self, slot, waiting):
        return self.frame[slot % len(self.frame)]


class SlottedALOHA(SlottedProtocol):
    "Random access: in every slot, each node with waiting messages transmits with probability `p`."

    def __init__(
        self,
        p: float = 0.5, # Probability of transmitting in a slot
        queue_size: int | None = None, # Messages waiting per sender (see `SlottedProtocol`)
        max_retries: int = 0, # Retransmissions of a lost message
        seed: int | None = None, # Seed of the random access
    ):
        super().__init__(queue_size, max_retries)
        self.p = p
        self.rng = np.random.default_rng(seed)

    def reset(self, seed: int | None = None) -> None:
        super().reset(seed)
        if seed is not None:
            self.rng = np.random.default_rng([seed, 2])

    def senders(self, slot, waiting):
        return [node for node in waiting if self.rng.random() < self.p]

In [ ]:
show_doc(SlottedProtocol.senders)

In [ ]:
show_doc(SlottedProtocol.feedback)

### Tests

In [ ]:
from comm_core.core import TransmissionResult

In [ ]:
def msg(s, r, payload=None):
    return Message(s, r, payload)

def sent(protocol):
    return sorted((t.sender, t.receiver, t.message.payload) for t in protocol.schedule(0.0))

# TDMA: a, b, c take turns; the others' messages wait
tdma = TDMAProtocol(['a', 'b', 'c'])
for s, r in (('a', 'b'), ('b', 'c'), ('c', 'a'), ('a', 'c')):
    tdma.submit(msg(s, r), 0.0)
assert sent(tdma) == [('a', 'b', None), ('a', 'c', None)]
assert sent(tdma) == [('b', 'c', None)] and tdma.waiting('c') == 1
assert sent(tdma) == [('c', 'a', None)] and sent(tdma) == []          # a's slot again: nothing waiting
# slots shared by nodes far apart (spatial reuse)
tdma = TDMAProtocol([('a', 'c'), 'b'])
for s in 'abc':
    tdma.submit(msg(s, 'x'), 0.0)
assert [t.sender for t in tdma.schedule(0.0)] == ['a', 'c'] and [t.sender for t in tdma.schedule(0.0)] == ['b']
# only the newest messages wait
tdma = TDMAProtocol(['a', 'b'], queue_size=1)
for i in range(3):
    tdma.submit(msg('b', 'a', i), 0.0)
assert sent(tdma) == [] and sent(tdma) == [('b', 'a', 2)] and tdma.dropped == 2
# retransmissions: lost messages go out again in the sender's next slot, at most max_retries times
tdma = TDMAProtocol(['a'], max_retries=2)
tdma.submit(msg('a', 'b', 'hi'), 0.0)
lost = lambda ts: [TransmissionResult(success=False, transmission=t) for t in ts]
for _ in range(3):
    ts = tdma.schedule(0.0)
    assert [t.message.payload for t in ts] == ['hi']
    tdma.feedback(lost(ts))
assert tdma.schedule(0.0) == []                                        # 1 try + 2 retries, then given up
tdma.reset()
assert tdma.slot == 0 and tdma.waiting('a') == 0

# slotted ALOHA: each node transmits with probability p, reproducibly
aloha = SlottedALOHA(p=0.3)
def run(aloha, seed):
    aloha.reset(seed)
    out = []
    for _ in range(200):
        for s in 'abcd':
            aloha.submit(msg(s, 'x'), 0.0)
        out.append(sorted({t.sender for t in aloha.schedule(0.0)}))
    return out
first = run(aloha, 0)
assert run(aloha, 0) == first and run(aloha, 1) != first
assert abs(np.mean([len(x) for x in first]) / 4 - 0.3) < 0.05
assert [t.sender for t in SlottedALOHA(p=1.0).schedule(0.0)] == []
p1 = SlottedALOHA(p=1.0); p1.submit(msg('a', 'b'), 0.0); assert len(p1.schedule(0.0)) == 1

### With interference

Four nodes on a line, all with messages for each other at every step, in a `Network` with an `InterferenceBackend`:

* `IdealProtocol`: everybody transmits at once, so nobody hears (half-duplex).
* `TDMAProtocol`: one sender per slot, so all its messages get through, a slot later for each turn waited.
* `SlottedALOHA`: in between, with collisions.

In [ ]:
from comm_core import Network, Node, FullMeshTopology, RayTracedFading, SystemLevelChannel, InterferenceBackend

In [ ]:
xs = np.arange(4)
d = np.abs(np.subtract.outer(xs, xs)).astype(float)
table = RayTracedFading(np.where(d > 0, 1e-3 / np.maximum(d, 1) ** 1.5, 0), [(x, 0) for x in xs])
nodes = ['a', 'b', 'c', 'd']
table.set_positions({n: (i, 0) for i, n in enumerate(nodes)})

def delivered(protocol, steps=40):
    ch = SystemLevelChannel(fading=table, noise_power=1e-10, per_model='threshold', snr_threshold_db=10)
    net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), protocol, InterferenceBackend(ch))
    net.reset(seed=0)
    for _ in range(steps):
        for s in nodes:
            for r in nodes:
                if s != r:
                    net.send(Message(s, r, None, size_bits=64))
        net.step()
    return net.metrics.summary().successful / (steps * 12)              # 12 messages per step

ideal, tdma, aloha = (delivered(IdealProtocol()), delivered(TDMAProtocol(nodes, queue_size=3)),
                      delivered(SlottedALOHA(p=0.25, queue_size=3, seed=0)))
assert ideal == 0.0 and 0.2 < tdma <= 0.25 and 0 < aloha < tdma

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()